In [2]:
!pip install tensorflow

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/332.0 MB 4.8 MB/s eta 0:01:10
   ---------------------------------------- 1.6/332.0 MB 4.2 MB/s eta 0:01:18
   ---------------------------------------- 2.1/332.0 MB 3.7 MB/s eta 0:01:31
   ---------------------------------------- 3.1/332.0 MB 4.4 MB/s eta 0:01:16
   ---------------------------------------- 3.9

In [3]:
# Importing the Libraies
from tensorflow import keras
import tensorflow as tf
import pandas as pd
import os
import re

In [11]:
# Loading the data
male_data = pd.read_csv('Indian-Male-Names.csv')
female_data = pd.read_csv('Indian-Female-Names.csv')

In [12]:
# let's prepare two helper functions for data cleaning and data processing
repl_list = ['s/o', 'd/o', 'w/o', '/', '&', ',', '-']

def clean_data(name):
    # Convert to string and lowercase
    name = str(name).lower()
    
    # Remove non-ASCII characters
    name = ''.join(i for i in name if ord(i) < 128).strip()
    
    # Replace specific patterns from repl_list
    for repl in repl_list:
        name = name.replace(repl, " ")
    
    # Remove email parts if present
    if '@' in name:
        pos = name.find('@')
        name = name[:pos].strip()
    
    # Split and strip extra spaces
    name = name.split(" ")
    name = " ".join([each.strip() for each in name if each.strip() != ""])
    
    return name


In [13]:
def remove_records(merged_data):
    merged_data['delete'] = 0
    merged_data.loc[merged_data['name'].str.find('with') != -1,'delete'] = 1
    merged_data.loc[merged_data['count_words']>=5,'delete']=1
    merged_data.loc[merged_data['count_words']==0,'delete']=1
    merged_data.loc[merged_data['name'].str.contains(r'\d') == True,'delete']=1
    cleaned_data = merged_data[merged_data.delete==0]
    return cleaned_data

In [15]:
merged_data = pd.concat((male_data,female_data),axis=0)
merged_data['name'] = merged_data['name'].apply(clean_data)
merged_data['count_words'] = merged_data['name'].str.split().apply(len)
cleaned_data = remove_records(merged_data)
indian_cleaned_data = cleaned_data[['name','count_words']].drop_duplicates(subset='name',keep='first')
indian_cleaned_data['label'] = 'indian'
len(indian_cleaned_data)

13663

In [18]:
!pip install Faker

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/1.9 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 6.5 MB/s eta 0:00:00


In [19]:
# For non-Indian names, there is a nifty package called Faker. This generates names from different regions
from faker import Faker

# Create Faker instance with US locale
fake = Faker('en_US')

# Generate a random name
print(fake.name())


Stephen Davis


In [28]:
# Now let’s build a neural network to classify nationalities using name
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.preprocessing import sequence
from keras.utils import to_categorical
import numpy as np
from sklearn.preprocessing import LabelEncoder
from keras.callbacks import Callback
np.random.seed(42)


In [24]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

In [31]:
def build_model(hidden_units,max_len,vocab_size):
    model = Sequential()
    model.add(LSTM(hidden_units,input_shape=(max_len,vocab_size)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    print(model.summary())
    return model

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.callbacks import Callback

# Example: Define model
def build_model(embedding_dim, max_len, vocab_size):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))  # binary classification example
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Example: Custom Callback
class myCallback(Callback):
    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
    
    def on_epoch_end(self, epoch, logs=None):
        val_loss, val_acc = self.model.evaluate(self.X_val, self.y_val, verbose=0)
        print(f"\nEpoch {epoch+1}: Val Accuracy = {val_acc:.4f}")

In [ ]:
# Usage
model = build_model(100, max_len, vocab_size)
model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    callbacks=[myCallback(X_test, y_test)]
)
